# Détection 3D des Façades de Bâtiments
## Fusion LiDAR + Images 2D avec YOLO

Ce notebook démontre l'utilisation du système de détection de façades (LiDAR + image 2D).


In [ ]:
import sys
from pathlib import Path

# Rendre les imports robustes quel que soit le dossier courant (root ou notebooks/)
project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Python executable:", sys.executable)

import numpy as np

# Matplotlib est optionnel (mais recommandé). Si tu vois une erreur ici,
# change le kernel sur "Python (fd)" puis Restart & Run All.
try:
    import matplotlib.pyplot as plt
    get_ipython().run_line_magic("matplotlib", "inline")
except ModuleNotFoundError as e:
    print("⚠️ matplotlib non trouvé dans ce kernel:", e)
    print("   -> Kernel > Change kernel > Python (fd)")


## 1. Chargement des modules


In [ ]:
# Imports du projet
# Si tu vois une erreur "No module named open3d" / "ultralytics", tu n'es pas sur le bon kernel.
# -> Kernel > Change kernel > Python (.venv) puis Restart & Run All

import importlib
import sys

for pkg in ("open3d", "ultralytics"):
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError as e:
        print("❌ Dépendance manquante dans ce kernel:", e)
        print("Python executable:", sys.executable)
        print("➡️  Solution recommandée: Kernel > Change kernel > Python (fd)")
        print("➡️  Sinon, installe dans CE kernel: python -m pip install", pkg)
        raise

from src.lidar_processing import LiDARProcessor
from utils.nuscenes_mini import find_pairs
from src.facade_detection import FacadeDetectionPipeline


In [ ]:
# Sélection automatique d'une paire (LiDAR + CAM_FRONT) depuis le dossier nuscmini/
# Assure-toi d'avoir: <project_root>/nuscmini/samples, sweeps, v1.0-mini

nusc_root = project_root / "nuscmini"

pairs = find_pairs(nusc_root, cam_channel="CAM_FRONT", limit=1)
if not pairs:
    raise RuntimeError(
        f"Aucune paire trouvée dans {nusc_root}. Vérifie que le dossier est bien extrait."
    )

pair = pairs[0]
lidar_file = pair.lidar_path
image_file = pair.cam_path

print("nuScenes root:", nusc_root)
print("LiDAR:", lidar_file)
print("Image:", image_file)


## 2. Pipeline complet

Utilisez le pipeline complet pour traiter vos données LiDAR et images.


In [ ]:
# Utiliser le pipeline complet sur la paire nuScenes sélectionnée
config_path = project_root / "config" / "pipeline_config.yaml"
output_dir = project_root / "results" / "notebook_nuscmini"

pipeline = FacadeDetectionPipeline(config=str(config_path))

# Recommandé pour nuScenes ici: désactiver la fusion (calibration du fichier YAML est un placeholder)
try:
    pipeline.config.setdefault("fusion", {})
    pipeline.config["fusion"]["enable_fusion"] = False
except Exception:
    pass

results = pipeline.process(
    lidar_file=str(lidar_file),
    image_file=str(image_file),
    output_dir=str(output_dir)
)

print("\nRésultats:")
print(f"  - Façades LiDAR: {len(results['lidar_facades'])}")
print(f"  - Détections YOLO: {len(results['yolo_detections'])}")
print(f"\nFichiers générés dans: {output_dir}")
print(f"- YOLO: {output_dir / 'yolo_detections.png'}")
print(f"- 3D : {output_dir / '3d_visualization.png'}")


## 3. Fusion LiDAR → Image (calibration nuScenes réelle)

Ici, on projette les points LiDAR sur l'image (CAM_FRONT) avec la calibration contenue dans `nuscmini/v1.0-mini/*.json`.

Résultat: une image overlay `lidar_overlay.png` dans `results/notebook_nuscmini/`.


In [ ]:
import cv2
import numpy as np

from utils.nuscenes_calib import compute_calibration_for_sample_data
from src.fusion import DataFusion

# Vérifier que find_pairs() nous a donné les tokens sample_data
if not getattr(pair, 'lidar_sample_data_token', None) or not getattr(pair, 'cam_sample_data_token', None):
    raise RuntimeError("Tokens sample_data manquants dans `pair`.")

calib = compute_calibration_for_sample_data(
    nusc_root=nusc_root,
    lidar_sd_token=pair.lidar_sample_data_token,
    cam_sd_token=pair.cam_sample_data_token,
)

# Charger l'image
img = cv2.imread(str(image_file))
if img is None:
    raise RuntimeError(f"Impossible de lire l'image: {image_file}")

# Charger le LiDAR (pcd.bin)
pcd = LiDARProcessor(voxel_size=0.0, normal_radius=0.2).load_point_cloud(str(lidar_file))
pts = np.asarray(pcd.points)

# Setup DataFusion avec calibration nuScenes
f = DataFusion(calibration_file=None)
f.camera_matrix = calib.K.astype(np.float32)
f.distortion_coeffs = np.zeros((5,), dtype=np.float32)
f.rotation_matrix = calib.R.astype(np.float32)
f.translation_vector = calib.t.astype(np.float32)

pts2d, valid = f.project_3d_to_2d(pts, return_mask=True)
h, w = img.shape[:2]
in_img = (pts2d[:, 0] >= 0) & (pts2d[:, 0] < w) & (pts2d[:, 1] >= 0) & (pts2d[:, 1] < h)
valid = valid & in_img

overlay = f.overlay_point_cloud_on_image(img, pts2d, valid_mask=valid, point_size=1)

out_overlay = output_dir / "lidar_overlay.png"
out_overlay.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(out_overlay), overlay)

print("Overlay sauvegardé:", out_overlay)
print("Points projetés valides:", int(valid.sum()), "/", len(pts))

